# 02 - Train and evaluate: clean replay, replay poisoning, teacher-weighted repair (Stages 3-4)
**Needs:** the outputs of `01_splits_experiences.ipynb` (e.g. `results_01.zip` attached as an input; cell 0 restores them to `/kaggle/working/nids_out/`): `clean_*.parquet`, `class_map.json`, and `<scenario>/experience_rids.npz`, `scaler.json`, `experience_manifest.json`. The class schedule is read from there; nothing is assumed here.

**What happens**
1. **E0:** train the Transformer on E0. Save it twice as two separate files: `T.pt` (frozen teacher, never trained again) and `A0.pt` (first accepted model).
2. Three **arms**, all starting from A0 with identical replay sampling:
   - `clean`: replay buffer B untouched.
   - `poisoned`: the attacker flips the stored label of a bounded fraction of ONE old class in B (to BENIGN by default). D, val, test and T are out of reach.
   - `poisoned_repair`: same poison; if the gate rejects C1, **one** repair R1.
3. **Each update E1, E2:** C1 = last accepted model + new head rows, trained on the new data plus B. The **gate** (protected val rows) checks old-class recall floors, new-class recall, the benign false-alarm rate and the new-class false-alarm rate.
4. **Repair R1** restarts from the last accepted model (A0 at E1). Replay rows whose stored label is a class T knows are weighted by T's probability of that label; other rows keep weight 1. T is only read, and its sha256 is checked at the end.
5. **Accepted** -> new model in service, and only then does B gain the new experience's rows. **Rejected** -> keep the previous model, B unchanged.
6. Test rows are scored only at the very end, for the report.

Embedding happens inside the model, per batch; nothing here embeds the dataset.

In [ ]:
import glob, json, os, subprocess, sys
from pathlib import Path

# Folder holding s0_audit.py etc. On Kaggle: the uploaded code Dataset.
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in glob.glob("/kaggle/input/**/s0_audit.py", recursive=True)), ".")
INPUT = os.environ.get("NIDS_INPUT", "/kaggle/input")   # where the CICDDoS2019 CSVs live
sys.path.insert(0, CODE_DIR)
from common import default_work_dir, find_files
OUT = default_work_dir()
print("code:", CODE_DIR, "| data:", INPUT, "| outputs:", OUT)

def run(script, *args):
    """Run one of our scripts and stream its output (same as `!python script args`)."""
    cmd = [sys.executable, "-u", str(Path(CODE_DIR) / script), *map(str, args)]
    print("$", " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:  # stream: training prints progress as it goes
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed (exit {p.returncode})")

## 0. Restore the prepared Stage 1/1b files (fresh session)
Attach `results_01.zip` as an input: either the Kaggle Dataset Kaggle makes from it (auto-extracted) or the zip itself. `/kaggle/input` is read-only and Stage 3 writes `runs/` next to the data, so the files are copied to `/kaggle/working/nids_out/`. Nothing is recomputed: no audit, no preprocessing.

In [ ]:
import shutil, zipfile
REQUIRED = ["split_manifest.json", "class_map.json", "protected_exclusions.json", "clean_train.parquet",
            "clean_val.parquet", "clean_test.parquet",
            *[f"{sc}/{n}" for sc in ("ci", "cii") for n in ("experience_rids.npz", "scaler.json", "experience_manifest.json")]]
complete = lambda d: all((Path(d) / n).exists() for n in REQUIRED)
if complete(OUT):
    print("prepared files already in", OUT)
else:
    dirs = [Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/protected_exclusions.json", recursive=True))]
    src = next((d for d in dirs if complete(d)), None)
    if src is not None:
        shutil.copytree(src, OUT, dirs_exist_ok=True)
        print("copied prepared files from", src)
    else:
        zips = sorted(glob.glob("/kaggle/input/**/results_01.zip", recursive=True))
        assert zips, "attach results_01.zip (or the Dataset made from it) as an input"
        with zipfile.ZipFile(zips[0]) as z:
            assert all(n.startswith("nids_out/") for n in z.namelist()), "unexpected archive layout"
            z.extractall(OUT.parent)
        print("extracted", zips[0], "into", OUT.parent)
missing = [n for n in REQUIRED if not (OUT / n).exists()]
assert not missing, f"missing prepared files: {missing}"
print("all", len(REQUIRED), "required files present")

## 1. Verify the prepared files and the GPU
`--dry-run` loads every input through the exact training code path and checks it against Stage 1b (experience rows, kept val/test rows, the 166-row exclusion hash), then stops. Nothing is trained or written.

In [ ]:
import torch
SCENARIO = "cii"   # or "ci"
em = json.loads((OUT / SCENARIO / "experience_manifest.json").read_text())
print("schedule (from Stage 1b):", em["schedule"], "| attack cap:", em["attack_cap"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU - slow)")
assert torch.cuda.is_available(), "switch the notebook accelerator to GPU T4"
for sc in ("cii", "ci"):
    run("s3_continual.py", "--scenario", sc, "--dry-run")

## 2. Settings
`POISON_TARGET` must be an E0 attack class (T must know it); `None` = first E0 attack class. `POISON_FRAC` is the bound on the share of that class's buffer rows the attacker may flip.
Gate defaults: old class may lose at most 0.05 recall, new class >= 0.80, benign FAR <= 0.05, new-class FAR <= 0.05. Report whatever you choose; don't tune them on test.

In [ ]:
RUN = "main"
POISON_TARGET = None
POISON_TO = "BENIGN"
POISON_FRAC = 0.5
MEM_PER_CLASS = 2000
EPOCHS0, EPOCHS_UPDATE = 10, 5
GATE = {"--gate-old-drop": 0.05, "--gate-new-min": 0.80, "--gate-far-max": 0.05, "--gate-newfar-max": 0.05}

args = ["--scenario", SCENARIO, "--poison-to", POISON_TO, "--poison-frac", POISON_FRAC,
        "--mem-per-class", MEM_PER_CLASS, *[str(x) for kv in GATE.items() for x in kv]]
if POISON_TARGET:
    args += ["--poison-target", POISON_TARGET]

## 3. Timing check (1 epoch everywhere, separate run name `quickcheck`)
Same data, arms and gate as the full run, so it also proves the whole pipeline runs on the GPU. Its accept/reject decisions and metrics are **not results** (1 epoch).

In [ ]:
import time
import pandas as pd
t0 = time.time()
run("s3_continual.py", *args, "--run", "quickcheck", "--epochs0", 1, "--epochs-update", 1)
wall = time.time() - t0
Q = OUT / "runs" / SCENARIO / "quickcheck"
e0 = json.loads((Q / "A0_metrics.json").read_text())["seconds"]
rows = [{"arm": arm, "E": r["experience"], "attempt": r["attempt"], "train_rows": r["n_current"] + r["n_replay"],
         "sec_1_epoch_plus_val": round(r["seconds"], 1)}
        for arm in ("clean", "poisoned", "poisoned_repair") for r in json.loads((Q / arm / "rounds.json").read_text())]
print(pd.DataFrame(rows).to_string(index=False))
upd = sum(r["sec_1_epoch_plus_val"] for r in rows)
print(f"\nE0: {e0:.1f}s for 1 epoch | wall time of the whole check: {wall:.0f}s")
print(f"rough full-run estimate (E0 x{EPOCHS0}, every attempt x{EPOCHS_UPDATE}): "
      f"~{(e0 * EPOCHS0 + upd * EPOCHS_UPDATE + (wall - e0 - upd)) / 60:.1f} min "
      "(more if the full run makes more R1 attempts than this check)")
print("T unchanged in quick check:", json.loads((Q / "run_manifest.json").read_text())["T_unchanged"])

## STOP - send the output of sections 1 and 3 (the verify cell and the timing-check cell) before the full run
Then set `QUICK_CHECK_REVIEWED = True` below.

## 4. Full run
`--resume` reuses saved T/A0 and skips finished arms, so if the Kaggle session dies, rerun this cell. It refuses to resume if any Stage 1/1b file or setting changed since the run started (then use a new `RUN` name).

In [ ]:
QUICK_CHECK_REVIEWED = False
assert QUICK_CHECK_REVIEWED, "STOP: send the timing-check output first, then set QUICK_CHECK_REVIEWED = True"
run("s3_continual.py", *args, "--run", RUN, "--epochs0", EPOCHS0, "--epochs-update", EPOCHS_UPDATE, "--resume")

## 5. Report: tables and plots (reads saved JSON only)

In [ ]:
import pandas as pd
run("s4_report.py", "--scenario", SCENARIO, "--run", RUN)
RDIR = OUT / "runs" / SCENARIO / RUN
att = pd.read_csv(RDIR / "report" / "attempts.csv")
print(att.to_string(index=False))
try:
    from IPython.display import Image, display
    for png in sorted((RDIR / "report").glob("*.png")):
        display(Image(filename=str(png)))
except ImportError:
    print("plots in", RDIR / "report")

## 6. Read it correctly
- `T_unchanged` must be `True`.
- `w_flipped` vs `w_clean`: the mean teacher weight on poisoned vs clean replay rows (diagnostic only; flips are never used in training). A low `w_flipped` means T recognised the bad labels.
- An update that no candidate passed leaves the old model in service, so its new classes show as `n/l`.
- Every false-alarm rate comes with `n_benign_val`. With few benign rows, the rate is coarse.

In [ ]:
man = json.loads((RDIR / "run_manifest.json").read_text())
cfg = json.loads((RDIR / "config.json").read_text())
print("T_unchanged:", man["T_unchanged"], "| T classes:", man["T_classes"])
print("same Stage 1/1b inputs as cell 1:", man["inputs"] == cfg["inputs"])
fin = pd.read_csv(RDIR / "report" / "final_test.csv")
print("all arms tested on the same rows:", fin["test_rows_sha256"].nunique() == 1, "| rows:", fin["n_test_rows"].iloc[0])
print(fin.drop(columns=["test_rows_sha256"]).T.to_string())
clean = att[att["arm"] == "clean"]
print("\nclean arm accepted every update:", bool(clean["accepted"].all()))
if not clean["accepted"].all():
    print("  -> the clean baseline itself failed the gate; report this before reading the poisoning arms:")
    print(clean[["experience", "gate_reasons"]].to_string(index=False))

## 7. Keep the results
`/kaggle/working` disappears when the session ends. This zips everything except the large parquet files (manifests, scaler, checkpoints, buffers, rounds, report). Download it from the Output panel.

In [ ]:
import zipfile
ZIP = OUT.parent / "nids_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p.suffix != ".parquet":
            z.write(p, p.relative_to(OUT.parent))
print(f"download {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB)")

## STOP 3 - what to bring back
The output of cells 5 and 6 (`attempts.csv`, `final_test.csv`, the checks above) and the PNGs. No result is claimed until this has been read.